In [1]:
import os
if os.name == 'nt' and 'HOME' not in os.environ:
    os.environ['HOME'] = os.path.expanduser("~")

from chop.models import get_model

model = get_model("chronos-2", pretrained=True)

c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\timm\models\helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
from chop import MaseGraph
import torch
import chop.passes as passes

batch_size = 1

print(model.config.chronos_config)
c_len = model.config.chronos_config["context_length"]
out_patch = model.config.chronos_config["output_patch_size"]

my_dummy_inputs = {
    "context": torch.randn((batch_size, c_len)),
    "group_ids": torch.zeros((batch_size,), dtype=torch.long),
    "future_covariates": torch.zeros((batch_size, out_patch), dtype=torch.float32),
    "num_output_patches": 1,
}

mg = MaseGraph(
    model,
    hf_input_names=[
        "context",
        "group_ids",
        "future_covariates",
        "num_output_patches",
    ],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": my_dummy_inputs})

`past_key_values` were not specified as input names, but model.config.use_cache = True. Setting model.config.use_cache = False.


{'context_length': 8192, 'input_patch_size': 16, 'input_patch_stride': 16, 'max_output_patches': 64, 'output_patch_size': 16, 'quantiles': [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99], 'time_encoding_scale': 8192, 'use_arcsinh': True, 'use_reg_token': True}
tensor([[-0.7663,  1.3306, -0.8697,  ...,  0.8374,  0.9261, -0.2293]])
tensor([[-0.7663,  1.3306, -0.8697,  ...,  0.8374,  0.9261, -0.2293]])
tensor([[False, False, False,  ..., False, False, False]])
tensor([[True, True, True,  ..., True, True, True]])
tensor([[-0.7663,  1.3306, -0.8697,  ...,  0.8374,  0.9261, -0.2293]])
tensor([[-0.7663,  1.3306, -0.8697,  ...,  0.8374,  0.9261, -0.2293]])
tensor([[-0.7626,  1.3344, -0.8660,  ...,  0.8411,  0.9298, -0.2256]])
tensor([[0.5816, 1.7805, 0.7499,  ..., 0.7075, 0.8645, 0.0509]])
tensor([[1.0105]])
tensor([[-0.7000,  1.0950, -0.7799,  ...,  0.7611,  0.8273, -0.2226]])
tensor([[-0.7000,  1.0950, -0.7799,  ...,  0.76

In [3]:
# Add chronos_config to mg.model
setattr(mg.model, "config", model.config)
setattr(mg.model, "chronos_config", model.chronos_config)

# The GraphModule returns a plain dict; wrap forward() to restore Chronos2Output type
from chop.models.chronos2.modeling_chronos2 import Chronos2Output

_orig_forward = mg.model.forward
def _patched_forward(*args, **kwargs):
    output = _orig_forward(*args, **kwargs)
    if isinstance(output, dict) and not isinstance(output, Chronos2Output):
        return Chronos2Output(**output)
    return output
mg.model.forward = _patched_forward


# Prediction Test

In [4]:
import pandas as pd
context_df = pd.read_csv("https://autogluon.s3.amazonaws.com/datasets/timeseries/m4_hourly/train.csv")
print("Input dataframe shape:", context_df.shape)
display(context_df.head())

Input dataframe shape: (353500, 3)


,item_id,timestamp,target
0,H1,1750-01-01 00:00:00,605.0
1,H1,1750-01-01 01:00:00,586.0
2,H1,1750-01-01 02:00:00,586.0
3,H1,1750-01-01 03:00:00,559.0
4,H1,1750-01-01 04:00:00,511.0


In [5]:
from chop.passes.module import report_trainable_parameters_analysis_pass

_, _ = report_trainable_parameters_analysis_pass(mg.model)

+----------------------------------------------------+------------------------+
| Submodule                                          |   Trainable Parameters |
+====================================================+========================+
| input_patch_embedding                              |                2548224 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.hidden_layer                 |                 150528 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.act                          |                      0 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.output_layer                 |                2360064 |
+----------------------------------------------------+------------------------+
| input_patch_embedding.dropout                      |                      0 |
+---------------------------------------

In [6]:
# Prepare pipeline for prediction
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline(model=mg.model)

pred_df = pipeline.predict_df(context_df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9])
print("Output dataframe shape:", pred_df.shape)
display(pred_df.head())


Output dataframe shape: (9936, 7)


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,H1,1750-01-30 04:00:00,target,637.280884,617.998718,637.280884,654.875305
1,H1,1750-01-30 05:00:00,target,577.113586,555.527100,577.113586,595.294922
2,H1,1750-01-30 06:00:00,target,536.170715,516.101318,536.170715,557.010986
3,H1,1750-01-30 07:00:00,target,499.121460,476.294617,499.121460,525.302734
4,H1,1750-01-30 08:00:00,target,472.389343,447.503540,472.389343,500.301514


In [7]:
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline(model=model)

pred_df = pipeline.predict_df(context_df, prediction_length=24, quantile_levels=[0.1, 0.5, 0.9])
print("Output dataframe shape:", pred_df.shape)
display(pred_df.head())

Output dataframe shape: (9936, 7)


,item_id,timestamp,target_name,predictions,0.1,0.5,0.9
0,H1,1750-01-30 04:00:00,target,611.081177,593.700012,611.081177,630.206421
1,H1,1750-01-30 05:00:00,target,554.298096,530.708496,554.298096,575.519897
2,H1,1750-01-30 06:00:00,target,506.850769,484.387268,506.850769,529.408936
3,H1,1750-01-30 07:00:00,target,474.666748,453.394714,474.666748,497.394318
4,H1,1750-01-30 08:00:00,target,453.156982,433.023560,453.156982,474.960968


# Perform Fev Bench

In [13]:
import fev
from chronos import Chronos2Pipeline

# Define the benchmark task
task = fev.Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="m4_hourly",
    horizon=24,
)

# Run evaluation with mg.model
pipeline = Chronos2Pipeline(model=mg.model)
predictions_per_window, inference_time_s = pipeline.predict_fev(task, batch_size=256)

print(f"Inference time: {inference_time_s:.2f}s")

# Compute evaluation metrics against the test split
summary = task.evaluation_summary(predictions_per_window, "test")
display(summary)


Inference time: 55.36s


{'model_name': 'test',
 'dataset_path': 'autogluon/chronos_datasets',
 'dataset_config': 'm4_hourly',
 'horizon': 24,
 'num_windows': 1,
 'initial_cutoff': -24,
 'window_step_size': 24,
 'min_context_length': 1,
 'max_context_length': None,
 'seasonality': 1,
 'eval_metric': 'MASE',
 'extra_metrics': [],
 'quantile_levels': [],
 'id_column': 'id',
 'timestamp_column': 'timestamp',
 'target': 'target',
 'generate_univariate_targets_from': None,
 'known_dynamic_columns': [],
 'past_dynamic_columns': [],
 'static_columns': [],
 'task_name': 'm4_hourly',
 'test_error': 5.747320368865849,
 'training_time_s': None,
 'inference_time_s': None,
 'num_forecasts': 414,
 'dataset_fingerprint': '19e36bb78b718d8d',
 'trained_on_this_dataset': False,
 'fev_version': '0.7.0',
 'MASE': 5.747320368865849}

In [16]:
fev.leaderboard(summaries=[summary], baseline_model="test")

c:\Users\johnn\Documents\coding\mase\.venv\Lib\site-packages\fev\analysis.py:553: RuntimeWarning: Mean of empty slice
  return np.nanmean(wins, axis=1)  # [n_models, ...]


,win_rate,skill_score,median_training_time_s,median_inference_time_s,training_corpus_overlap,num_failures
model_name,,,,,,
test,NaN,0.0,NaN,NaN,0.0,0
